In [1]:
import sys, os
# ensure parent directory is on the path so `src` package can be imported
sys.path.insert(0, os.path.abspath('..'))

In [2]:
# configura per importare da src
import sys
sys.path.append('./src')

In [3]:
from src.model import ConceptBoxModel
from src.train import train_step
from src.dataset import load_and_split_awa2_features
from torch import optim
import torch
import numpy as np
import pandas as pd

In [ ]:
# leggi features da file txt
with open('./data/features.txt', 'r') as f:
    h_features = np.array([[float(num) for num in line.split()] for line in f])
h_features = torch.tensor(h_features, dtype=torch.float32)

In [6]:
# leggi matrice V_gt da file csv
V_gt = pd.read_csv('../AwA2_Dataset_Labels/Animals_with_Attributes2/V_gt.csv', sep='\t', index_col=0).values
V_gt = torch.tensor(V_gt, dtype=torch.float32)

In [ ]:
NUM_CLASSES = V_gt.shape[0]  # numero di classi è dato dal numero di righe di V_gt
NUM_CONCEPTS = V_gt.shape[1]  # numero di concetti è dato dal numero di colonne di V_gt 

torch.Size([51, 51])

In [15]:
V_gt = torch.rand(32, 50, 50)
V_gt.shape

torch.Size([32, 50, 50])

In [7]:
features_path = '../AwA2_Dataset_Features/Animals_with_Attributes2/Features/ResNet101/AwA2-features.txt'
labels_path = '../AwA2_Dataset_Features/Animals_with_Attributes2/Features/ResNet101/AwA2-labels.txt'
classes_path = '../Awa2_Dataset_Labels/Animals_with_Attributes2/classes.txt'
train_split_path = '../AwA2_Dataset_Features/Animals_with_Attributes2/Features/ResNet101/trainclasses1.txt'
val_split_path = '../AwA2_Dataset_Features/Animals_with_Attributes2/Features/ResNet101/valclasses1.txt'
test_split_path = '../AwA2_Dataset_Features/Animals_with_Attributes2/Features/ResNet101/testclasses.txt'

(X_train, y_train), (X_val, y_val), (X_test, y_test) = load_and_split_awa2_features(
    features_path, labels_path, classes_path, 
    train_split_path, val_split_path, test_split_path
)

Caricamento in corso... (potrebbe richiedere qualche secondo)
Dimensioni totali feature: (37322, 2048)
Training set: 20218 samples (27 classi)
Validation set: 9191 samples (13 classi)
Test set: 7913 samples (10 classi)


In [10]:
y_train[4002]

np.int64(49)

In [ ]:
# Iperparametri fittizi basati sull'architettura
LATENT_DIM = 512
BOX_DIM = 16
BATCH_SIZE = 32
    
model = ConceptBoxModel(LATENT_DIM, NUM_CONCEPTS, BOX_DIM, NUM_CLASSES)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
    
# Dati fittizi (simulano le features estratte da Psi(x) e le ground truth)
h_features = torch.randn(BATCH_SIZE, LATENT_DIM)
labels = torch.randint(0, NUM_CLASSES, (BATCH_SIZE,))
concepts = torch.randint(0, 2, (BATCH_SIZE, NUM_CONCEPTS)).float()
    
# Matrice V ground truth fittizia calcolata a priori (es. da WordNet)
V_gt = torch.rand(BATCH_SIZE, NUM_CONCEPTS, NUM_CONCEPTS)
    
# Esecuzione di uno step di training
total, l_t, l_c, l_b = train_step(model, optimizer, h_features, labels, concepts, V_gt)
    
print(f"Loss Totale: {total:.4f} (Task: {l_t:.4f}, Concept: {l_c:.4f}, Box: {l_b:.4f})")

Loss Totale: 4.9698 (Task: 3.9370, Concept: 0.6988, Box: 0.3339)
